# Latency, explanation controls, evidence agent and implementation audit
Run cells in order. Durable artifacts are written to the configured Drive results folder.


In [ ]:
from pathlib import Path
import os, sys, json, errno, time
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None

def with_drive_retry(operation, attempts=3):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errno.ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or attempt == attempts - 1:
                raise
            print(f"Drive disconnected; remounting before retry {attempt + 1}/{attempts - 1}.", flush=True)
            if drive is not None:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2():
    configured = os.environ.get("ATLAS_V2_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [
        DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
        Path.cwd(), *Path.cwd().parents,
    ]
    for candidate in candidates:
        if all((candidate/name).is_file() for name in (
            "src/atlas_v2/data.py", "src/atlas_v2/models.py",
            "src/atlas_v2/experiments.py", "src/atlas_v2/agent.py", "requirements.txt"
        )):
            return candidate.resolve()
    raise FileNotFoundError("Synchronized implementation/v2 source is not visible to this kernel.")

V2 = with_drive_retry(resolve_v2)
sys.path.insert(0, str(V2/"src"))
for module in list(sys.modules):
    if module == "atlas_v2" or module.startswith("atlas_v2."):
        del sys.modules[module]
import atlas_v2
print("Code:", V2)


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
import numpy as np, pandas as pd, sklearn, scipy, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CPU count:", os.cpu_count())


In [ ]:
RESULTS = DRIVE/"projects/implementation/ATLAS-IDNet/results/v2" if DRIVE.exists() else V2/"results"
PUBLICATION_CONFIG_PATH = RESULTS/"publication_config.json"
PUBLICATION_CONFIG = json.loads(PUBLICATION_CONFIG_PATH.read_text())
print(json.dumps(PUBLICATION_CONFIG, indent=2))


In [ ]:
from atlas_v2.experiments import diagnostics, summarize
main_runs = []
for dataset_name, dataset_protocols in PUBLICATION_CONFIG["protocols"].items():
    for protocol_name, spec in dataset_protocols.items():
        root = RESULTS/dataset_name/"publication"/protocol_name
        candidates = []
        for complete in root.glob("atlas_*/complete.json"):
            payload = json.loads(complete.read_text())
            if payload["config"]["seed"] == 42 and payload["config"]["loss"] == PUBLICATION_CONFIG["loss"]:
                candidates.append(complete.parent)
        if len(candidates) != 1:
            raise RuntimeError(f"Expected exactly one seed-42 main run for {dataset_name}/{protocol_name}; found {len(candidates)}")
        run_dir = candidates[0]
        diagnostics(spec["prepared"], run_dir, samples=256, repeats=100)
        main_runs.append(run_dir)
        summarize(root)
print("Diagnosed main runs:", *main_runs, sep="\n")


In [ ]:
from atlas_v2.agent import evaluate_agent
agent_results = evaluate_agent(main_runs, RESULTS/"agent_evaluation")
display(agent_results)
if not agent_results["complete"].all():
    raise RuntimeError("At least one real-run agent evaluation is incomplete.")
if not (agent_results["unsupported_claim_rate"] == 0).all():
    raise RuntimeError("The evidence agent emitted unsupported claims.")


In [ ]:
rows = []
for dataset_name in PUBLICATION_CONFIG["protocols"]:
    target = PUBLICATION_CONFIG["submitted_reference"][dataset_name]
    for protocol_name in ("natural", "minority_challenge"):
        all_runs = pd.read_csv(RESULTS/dataset_name/"publication"/protocol_name/"all_runs.csv")
        main = all_runs[(all_runs.variant == "atlas") & (all_runs.loss == PUBLICATION_CONFIG["loss"])]
        rows.append({
            "dataset": dataset_name, "protocol": protocol_name, "completed_main_seeds": main.seed.nunique(),
            "accuracy_mean": main.accuracy.mean(), "weighted_f1_mean": main.weighted_f1.mean(),
            "macro_f1_mean": main.macro_f1.mean(), "mcc_mean": main.mcc.mean(),
            "submitted_accuracy_target": target["accuracy"],
            "submitted_weighted_f1_target": target["weighted_f1"],
            "accuracy_above_submitted_target": bool(main.accuracy.mean() > target["accuracy"]),
            "weighted_f1_above_submitted_target": bool(main.weighted_f1.mean() > target["weighted_f1"]),
        })
audit = pd.DataFrame(rows)
audit.to_csv(RESULTS/"final_implementation_audit.csv", index=False)
display(audit)
print("Target comparisons are descriptive. They do not permit test-set tuning or result manipulation.")
